# Synthetic Philadelphia - Reworked Pipeline

This version caches the fully rasterized city and parallelizes the per-agent workflow: destination diary generation, dense trajectory generation, sparse sampling, and dataframe extraction.

`generate_dest_diary` uses optimization #2 (NumPy arrays for the EPR inner loop). Benchmark against baseline in `golden_dest_diary_benchmark.py`.

City pickle caches baked gravity data (`restore_gravity` on load). Rebuild with `REBUILD_CITY_CACHE = True` after changing the city.

In [ ]:
from pathlib import Path
import json
import time

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
import pyarrow.dataset as ds
from joblib import Parallel, cpu_count, delayed
from shapely.geometry import box

import nomad.map_utils as nm
import importlib
import nomad.city_gen
importlib.reload(nomad.city_gen)
from nomad.city_gen import RasterCity, load as load_city_pickle, save as save_city_pickle
from nomad.io.base import from_df
from nomad.traj_gen import Agent, Population

## Configuration

In [ ]:
LARGE_BOX = box(-75.1905, 39.9235, -75.1425, 39.9535)

USE_FULL_CITY = False
OUTPUT_DIR = Path("output_rework")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ox.settings.cache_folder = "cache_rework"

if USE_FULL_CITY:
    BOX_NAME = "full"
    POLY = "Philadelphia, Pennsylvania, USA"
else:
    BOX_NAME = "large"
    POLY = LARGE_BOX

SPATIAL_GPKG = OUTPUT_DIR / f"spatial_data_{BOX_NAME}.gpkg"
ROTATION_METADATA_PATH = OUTPUT_DIR / f"rotation_metadata_{BOX_NAME}.json"
CITY_CACHE_PICKLE = OUTPUT_DIR / f"raster_city_{BOX_NAME}.pkl"

REGENERATE_SPATIAL_DATA = False
REBUILD_CITY_CACHE = False
N_JOBS = -1

config = {
    "box_name": BOX_NAME,
    "block_side_length": 15.0,
    "hub_size": 100,
    "N": 200,
    "name_seed": 42,
    "name_count": 2,
    "epr_params": {
        "datetime": "2025-05-23 00:00-05:00",
        "end_time": "2025-07-01 00:00-05:00",
        "epr_time_res": 15,
        "rho": 0.4,
        "gamma": 0.3,
        "seed_base": 100,
    },
    "traj_params": {
        "dt": 0.5,
        "seed_base": 200,
    },
    "sampling_params": {
        "beta_ping": 7,
        "beta_start": 300,
        "beta_durations": 55,
        "ha": 11.5 / 15,
        "seed_base": 1,
    },
}

## Helpers

In [ ]:
def section(title):
    print("\n" + "=" * 50)
    print(title)
    print("=" * 50)


def write_parquet_file(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)


def prepare_traj_dataset(df):
    out = from_df(df, mixed_timezone_behavior="naive")
    out["datetime"] = out["datetime"].astype(str)
    return out


def write_dataset_from_files(paths, path, partition_cols):
    dataset = ds.dataset([str(file_path) for file_path in paths], format="parquet")
    ds.write_dataset(
        dataset.scanner(),
        base_dir=str(path),
        format="parquet",
        partitioning=partition_cols,
        existing_data_behavior="delete_matching",
    )


def write_traj_dataset_from_files(paths, path):
    dataset = ds.dataset([str(file_path) for file_path in paths], format="parquet")
    ds.write_dataset(
        dataset.scanner(),
        base_dir=str(path),
        format="parquet",
        partitioning=["date"],
        partitioning_flavor="hive",
        existing_data_behavior="delete_matching",
    )

## Spatial Data Cache

In [ ]:
section("SPATIAL DATA")

if REGENERATE_SPATIAL_DATA or not SPATIAL_GPKG.exists():
    t0 = time.time()
    buildings = nm.download_osm_buildings(
        POLY,
        crs="EPSG:3857",
        schema="garden_city",
        clip=True,
        infer_building_types=True,
        explode=True,
    )
    print(f"Buildings download: {time.time() - t0:>6.2f}s ({len(buildings):,} buildings)")

    if USE_FULL_CITY:
        boundary_polygon = nm.get_city_boundary_osm(POLY, simplify=True)[0]
        boundary_polygon = gpd.GeoSeries([boundary_polygon], crs="EPSG:4326").to_crs("EPSG:3857").iloc[0]
    else:
        boundary_polygon = gpd.GeoDataFrame(geometry=[POLY], crs="EPSG:4326").to_crs("EPSG:3857").geometry.iloc[0]

    buildings = gpd.clip(buildings, gpd.GeoDataFrame(geometry=[boundary_polygon], crs="EPSG:3857"))
    buildings = nm.remove_overlaps(buildings).reset_index(drop=True)

    t1 = time.time()
    streets = nm.download_osm_streets(
        POLY,
        crs="EPSG:3857",
        clip=True,
        explode=True,
        graphml_path=OUTPUT_DIR / "streets_consolidated.graphml",
    ).reset_index(drop=True)
    print(f"Streets download:   {time.time() - t1:>6.2f}s ({len(streets):,} streets)")

    t2 = time.time()
    rotated_streets, rotation_deg = nm.rotate_streets_to_align(streets, k=200)
    all_streets = streets.geometry.union_all()
    rotation_origin = (all_streets.centroid.x, all_streets.centroid.y)
    rotated_buildings = nm.rotate(buildings, rotation_deg=rotation_deg, origin=rotation_origin)
    rotated_boundary = nm.rotate(
        gpd.GeoDataFrame(geometry=[boundary_polygon], crs="EPSG:3857"),
        rotation_deg=rotation_deg,
        origin=rotation_origin,
    )
    print(f"Grid rotation:      {time.time() - t2:>6.2f}s ({rotation_deg:.2f} deg)")

    if SPATIAL_GPKG.exists():
        SPATIAL_GPKG.unlink()
    rotated_buildings.to_file(SPATIAL_GPKG, layer="buildings", driver="GPKG")
    rotated_streets.to_file(SPATIAL_GPKG, layer="streets", driver="GPKG", mode="a")
    rotated_boundary.to_file(SPATIAL_GPKG, layer="boundary", driver="GPKG", mode="a")

    with open(ROTATION_METADATA_PATH, "w") as f:
        json.dump({"rotation_deg": rotation_deg, "rotation_origin": rotation_origin}, f)
else:
    print(f"Loading spatial data from {SPATIAL_GPKG}")

with open(ROTATION_METADATA_PATH) as f:
    rotation_metadata = json.load(f)

rotation_deg = rotation_metadata["rotation_deg"]
rotation_origin = tuple(rotation_metadata["rotation_origin"])


SPATIAL DATA
Loading spatial data from output_rework/spatial_data_large.gpkg


## Raster City Cache

In [ ]:
section("RASTER CITY CACHE")

if REBUILD_CITY_CACHE or not CITY_CACHE_PICKLE.exists():
    buildings = gpd.read_file(SPATIAL_GPKG, layer="buildings")
    streets = gpd.read_file(SPATIAL_GPKG, layer="streets")
    boundary = gpd.read_file(SPATIAL_GPKG, layer="boundary")

    t0 = time.time()
    city_for_cache = RasterCity(
        boundary.geometry.iloc[0],
        streets,
        buildings,
        block_side_length=config["block_side_length"],
        resolve_overlaps=True,
        other_building_behavior="filter",
        rotation_deg=rotation_deg,
        rotation_origin=rotation_origin,
    )
    print(f"City generation:    {time.time() - t0:>6.2f}s")

    t1 = time.time()
    city_for_cache.get_street_graph()
    print(f"Street graph:       {time.time() - t1:>6.2f}s")

    t2 = time.time()
    city_for_cache._build_hub_network(hub_size=config["hub_size"])
    print(f"Hub network:        {time.time() - t2:>6.2f}s")

    t3 = time.time()
    city_for_cache.compute_gravity(exponent=2.0, callable_only=True)
    print(f"Gravity:            {time.time() - t3:>6.2f}s")

    t4 = time.time()
    city_for_cache.compute_shortest_paths(callable_only=True)
    print(f"Shortest paths:     {time.time() - t4:>6.2f}s")

    city_for_cache.grav = None
    city_for_cache.grav_for_candidates = None
    city_for_cache.shortest_paths = None

    save_city_pickle(city_for_cache, CITY_CACHE_PICKLE)
    print(f"Saved raster city cache to {CITY_CACHE_PICKLE}")
else:
    print(f"Loading raster city cache from {CITY_CACHE_PICKLE}")


RASTER CITY CACHE
Loading raster city cache from output_rework/raster_city_large.pkl


In [ ]:
def load_city_from_cache():
    city = load_city_pickle(CITY_CACHE_PICKLE)
    city.rotation_deg = rotation_metadata["rotation_deg"]
    city.rotation_origin = tuple(rotation_metadata["rotation_origin"])
    if getattr(city, "mh_dist_nearby_doors", None) is not None and len(city.mh_dist_nearby_doors) > 0:
        if city.grav is None:
            if hasattr(city, "restore_gravity"):
                city.restore_gravity(exponent=2.0, callable_only=True)
            else:
                city.compute_gravity(exponent=2.0, callable_only=True)
    elif city.grav is None:
        city.compute_gravity(exponent=2.0, callable_only=True)
    if city.shortest_paths is None:
        city.compute_shortest_paths(callable_only=True)
    return city


city = load_city_from_cache()

summary_df = pd.DataFrame({
    "component": ["blocks", "streets", "buildings", "hubs", "nearby_door_pairs"],
    "count": [
        len(city.blocks_gdf),
        len(city.streets_gdf),
        len(city.buildings_gdf),
        len(city.hubs),
        len(city.mh_dist_nearby_doors),
    ],
})
print(summary_df.to_string(index=False))
print(city.buildings_gdf.building_type.value_counts())

        component   count
           blocks  104188
          streets   28316
        buildings   19960
             hubs     100
nearby_door_pairs 1019689
building_type
home         19306
retail         437
workplace      128
park            89
Name: count, dtype: int64


## Agent Parameters

In [ ]:
section("AGENT PARAMETERS")

population_template = Population(city)
population_template.generate_agents(
    N=config["N"],
    seed=config["name_seed"],
    name_count=config["name_count"],
    datetimes=config["epr_params"]["datetime"],
)

agents = list(population_template.roster.values())

agent_params = pd.DataFrame({
    "identifier": [agent.identifier for agent in agents],
    "home": [agent.home for agent in agents],
    "workplace": [agent.workplace for agent in agents],
    "datetime": [config["epr_params"]["datetime"]] * len(agents),
    "dest_seed": config["epr_params"]["seed_base"] + np.arange(len(agents)),
    "traj_seed": config["traj_params"]["seed_base"] + np.arange(len(agents)),
    "sampling_seed": config["sampling_params"]["seed_base"] + np.arange(len(agents)),
})

agent_params["beta_start"] = config["sampling_params"]["beta_start"]
agent_params["beta_ping"] = config["sampling_params"]["beta_ping"]
agent_params["beta_durations"] = config["sampling_params"]["beta_durations"]
agent_params["ha"] = config["sampling_params"]["ha"]

agent_params_path = OUTPUT_DIR / f"agent_params_{BOX_NAME}.parquet"
agent_params.to_parquet(agent_params_path, index=False)
agent_params.head()


AGENT PARAMETERS


,identifier,home,workplace,datetime,dest_seed,traj_seed,sampling_seed,beta_start,beta_ping,beta_durations,ha
0,clever_colden,h-x99-y33,w-x267-y257,2025-05-23 00:00-05:00,100,200,1,300,7,55,0.766667
1,clever_noether,h-x243-y251,w-x239-y293,2025-05-23 00:00-05:00,101,201,2,300,7,55,0.766667
2,cocky_clarke,h-x341-y127,w-x164-y185,2025-05-23 00:00-05:00,102,202,3,300,7,55,0.766667
3,cocky_panini,h-x145-y53,w-x276-y268,2025-05-23 00:00-05:00,103,203,4,300,7,55,0.766667
4,compassionate_davinci,h-x120-y182,w-x229-y129,2025-05-23 00:00-05:00,104,204,5,300,7,55,0.766667


## Per-Agent Generation

In [ ]:
end_time = pd.Timestamp(config["epr_params"]["end_time"])


def effective_n_jobs(n_jobs, task_count):
    """Return the number of worker batches for the configured joblib setting."""
    if n_jobs < 0:
        n_jobs = max(cpu_count() + 1 + n_jobs, 1)
    return min(n_jobs, task_count)


def simulate_agent(row, city_worker):
    t_agent = time.perf_counter()
    t0 = time.perf_counter()
    agent = Agent(
        identifier=row.identifier,
        city=city_worker,
        home=row.home,
        workplace=row.workplace,
        datetime=row.datetime,
    )
    init_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    agent.generate_dest_diary(
        end_time=end_time,
        epr_time_res=config["epr_params"]["epr_time_res"],
        rho=config["epr_params"]["rho"],
        gamma=config["epr_params"]["gamma"],
        seed=int(row.dest_seed),
    )
    dest_time = time.perf_counter() - t0
    dest_diary = agent.destination_diary.copy()
    dest_diary["identifier"] = row.identifier

    t0 = time.perf_counter()
    agent.generate_trajectory(
        dt=config["traj_params"]["dt"],
        seed=int(row.traj_seed),
    )
    traj_time = time.perf_counter() - t0
    full_points = len(agent.trajectory)

    t0 = time.perf_counter()
    agent.sample_trajectory(
        beta_start=float(row.beta_start),
        beta_durations=float(row.beta_durations),
        beta_ping=float(row.beta_ping),
        ha=float(row.ha),
        seed=int(row.sampling_seed),
        replace_sparse_traj=True,
        cache_traj=True,
    )
    sample_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    sparse = agent.sparse_traj.reset_index(drop=True).copy()
    diary = agent.diary.copy()
    extract_time = time.perf_counter() - t0
    summary = {
        "user_id": row.identifier,
        "dest_entries": len(dest_diary),
        "diary_entries": len(diary),
        "full_points": full_points,
        "sparse_points": len(sparse),
        "beta_start": row.beta_start,
        "beta_ping": row.beta_ping,
        "beta_durations": row.beta_durations,
        "ha": row.ha,
        "init_time": init_time,
        "dest_time": dest_time,
        "traj_time": traj_time,
        "sample_time": sample_time,
        "extract_time": extract_time,
        "agent_time": time.perf_counter() - t_agent,
    }
    return sparse, diary, dest_diary, summary


def simulate_agent_batch(batch_id, agent_batch, batch_output_dir):
    """Simulate one dataframe chunk and write its outputs to batch-local files."""
    t_batch = time.perf_counter()
    t0 = time.perf_counter()
    city_worker = load_city_from_cache()
    city_load_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    results = [simulate_agent(row, city_worker) for row in agent_batch.itertuples(index=False)]
    agent_loop_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    sparse_parts, diary_parts, dest_diary_parts, summary_rows = zip(*results)

    sparse_df = pd.concat(sparse_parts, ignore_index=True)
    diaries_df = pd.concat(diary_parts, ignore_index=True)
    dest_diaries_df = pd.concat(dest_diary_parts, ignore_index=True)
    summary_df = pd.DataFrame(summary_rows)
    concat_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    poi_data = pd.DataFrame({
        "building_id": city_worker.buildings_gdf["id"].values,
        "x": (city_worker.buildings_gdf["door_cell_x"].astype(float) + 0.5).values,
        "y": (city_worker.buildings_gdf["door_cell_y"].astype(float) + 0.5).values,
    })

    sparse_df = city_worker.to_mercator(sparse_df)
    diaries_df = diaries_df.merge(poi_data, left_on="location", right_on="building_id", how="left")
    diaries_df = diaries_df.drop(columns=["building_id"])
    diaries_df = city_worker.to_mercator(diaries_df)

    sparse_df["date"] = pd.to_datetime(sparse_df["timestamp"], unit="s").dt.date.astype(str)
    diaries_df["date"] = pd.to_datetime(diaries_df["timestamp"], unit="s").dt.date.astype(str)
    dest_diaries_df["date"] = pd.to_datetime(dest_diaries_df["datetime"]).dt.date.astype(str)
    summary_df["date"] = pd.to_datetime(config["epr_params"]["datetime"]).date().isoformat()
    reproject_time = time.perf_counter() - t0

    batch_dir = batch_output_dir / f"batch_{batch_id:04d}"
    paths = {
        "sparse": batch_dir / "sparse.parquet",
        "diaries": batch_dir / "diaries.parquet",
        "sparse_traj": batch_dir / "sparse_traj.parquet",
        "diaries_traj": batch_dir / "diaries_traj.parquet",
        "dest_diaries": batch_dir / "dest_diaries.parquet",
        "summary": batch_dir / "summary.parquet",
    }
    t0 = time.perf_counter()
    write_parquet_file(sparse_df, paths["sparse"])
    write_parquet_file(diaries_df, paths["diaries"])
    write_parquet_file(prepare_traj_dataset(sparse_df), paths["sparse_traj"])
    write_parquet_file(prepare_traj_dataset(diaries_df), paths["diaries_traj"])
    write_parquet_file(dest_diaries_df, paths["dest_diaries"])
    write_parquet_file(summary_df, paths["summary"])
    write_time = time.perf_counter() - t0

    counts = summary_df[["full_points", "sparse_points", "dest_entries", "diary_entries"]].sum()
    return {
        "batch_id": batch_id,
        "agents": len(agent_batch),
        "full_points": int(counts["full_points"]),
        "sparse_points": int(counts["sparse_points"]),
        "dest_entries": int(counts["dest_entries"]),
        "diary_entries": int(counts["diary_entries"]),
        "city_load_time": city_load_time,
        "agent_loop_time": agent_loop_time,
        "concat_time": concat_time,
        "reproject_time": reproject_time,
        "write_time": write_time,
        "batch_time": time.perf_counter() - t_batch,
        "paths": {key: str(value) for key, value in paths.items()},
    }

In [ ]:
section("PARALLEL AGENT GENERATION")

n_workers = effective_n_jobs(N_JOBS, len(agent_params))
agent_batches = [agent_params.iloc[idx] for idx in np.array_split(np.arange(len(agent_params)), n_workers)]
run_id = pd.Timestamp.now('UTC').strftime("%Y%m%d_%H%M%S")
batch_output_dir = OUTPUT_DIR / f"batch_outputs_{BOX_NAME}" / run_id

t0 = time.time()
batch_results = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(simulate_agent_batch)(batch_id, batch, batch_output_dir)
    for batch_id, batch in enumerate(agent_batches)
)
generation_time = time.time() - t0

batch_manifest = pd.DataFrame([
    {"batch_id": result["batch_id"], "output": key, "path": value}
    for result in batch_results
    for key, value in result["paths"].items()
])
batch_timing = pd.DataFrame([
    {key: result[key] for key in [
        "batch_id", "agents", "city_load_time", "agent_loop_time",
        "concat_time", "reproject_time", "write_time", "batch_time",
    ]}
    for result in batch_results
])
generation_summary = pd.concat(
    [pd.read_parquet(result["paths"]["summary"]) for result in batch_results],
    ignore_index=True,
)

print(f"Generated {len(agent_params)} agents in {generation_time:.2f}s")
print(f"Batch outputs: {batch_output_dir}")
print(generation_summary[["full_points", "sparse_points", "dest_entries", "diary_entries"]].sum())
print(batch_timing.drop(columns=["batch_id", "agents"]).sum().sort_values(ascending=False))
generation_summary.head()


PARALLEL AGENT GENERATION


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 15 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of  15 | elapsed:  4.3min remaining: 28.0min
[Parallel(n_jobs=-1)]: Done   4 out of  15 | elapsed:  4.4min remaining: 12.1min
[Parallel(n_jobs=-1)]: Done   6 out of  15 | elapsed:  4.4min remaining:  6.6min
[Parallel(n_jobs=-1)]: Done   8 out of  15 | elapsed:  4.4min remaining:  3.9min
[Parallel(n_jobs=-1)]: Done  10 out of  15 | elapsed:  4.5min remaining:  2.3min
[Parallel(n_jobs=-1)]: Done  12 out of  15 | elapsed:  4.5min remaining:  1.1min


Generated 200 agents in 278.86s
Batch outputs: output_rework/batch_outputs_large/20260619_153502
full_points      22464200
sparse_points      239023
dest_entries        80205
diary_entries       91971
dtype: int64


[Parallel(n_jobs=-1)]: Done  15 out of  15 | elapsed:  4.6min finished


,user_id,dest_entries,diary_entries,full_points,sparse_points,beta_start,beta_ping,beta_durations,ha,date
0,clever_colden,376,399,112321,1138,300,7,55,0.766667,2025-05-23
1,clever_noether,394,577,112321,1067,300,7,55,0.766667,2025-05-23
2,cocky_clarke,394,470,112321,986,300,7,55,0.766667,2025-05-23
3,cocky_panini,408,507,112321,1078,300,7,55,0.766667,2025-05-23
4,compassionate_davinci,425,487,112321,1134,300,7,55,0.766667,2025-05-23


## Save Final Datasets

In [ ]:
section("SAVE FINAL DATASETS")

config_path = OUTPUT_DIR / f"config_{BOX_NAME}.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

manifest_path = OUTPUT_DIR / f"batch_manifest_{BOX_NAME}.parquet"
batch_manifest.to_parquet(manifest_path, index=False)
timing_path = OUTPUT_DIR / f"batch_timing_{BOX_NAME}.parquet"
batch_timing.to_parquet(timing_path, index=False)

paths_by_output = batch_manifest.groupby("output")["path"].apply(list)
write_traj_dataset_from_files(paths_by_output["sparse_traj"], OUTPUT_DIR / f"sparse_traj_{BOX_NAME}")
write_traj_dataset_from_files(paths_by_output["diaries_traj"], OUTPUT_DIR / f"diaries_{BOX_NAME}")
write_dataset_from_files(paths_by_output["dest_diaries"], OUTPUT_DIR / f"dest_diaries_{BOX_NAME}", ["date"])
write_dataset_from_files(paths_by_output["summary"], OUTPUT_DIR / f"homes_{BOX_NAME}", ["date"])

write_dataset_from_files(paths_by_output["sparse"], OUTPUT_DIR / "device_level", ["date"])
write_dataset_from_files(paths_by_output["diaries"], OUTPUT_DIR / "travel_diaries", ["date"])

print(f"Config saved to {config_path}")
print(f"Agent params saved to {agent_params_path}")
print(f"Batch manifest saved to {manifest_path}")
print(f"Batch timing saved to {timing_path}")
print(f"Sparse trajectories: {OUTPUT_DIR / f'sparse_traj_{BOX_NAME}'}")
print(f"Realized diaries:    {OUTPUT_DIR / f'diaries_{BOX_NAME}'}")
print(f"Destination diaries: {OUTPUT_DIR / f'dest_diaries_{BOX_NAME}'}")


SAVE FINAL DATASETS
Config saved to output_rework/config_large.json
Agent params saved to output_rework/agent_params_large.parquet
Batch manifest saved to output_rework/batch_manifest_large.parquet
Sparse trajectories: output_rework/sparse_traj_large
Realized diaries:    output_rework/diaries_large
Destination diaries: output_rework/dest_diaries_large


## Quick Checks

In [ ]:
print("Sparse rows:", int(generation_summary["sparse_points"].sum()))
print("Diary rows:", int(generation_summary["diary_entries"].sum()))
print("Destination diary rows:", int(generation_summary["dest_entries"].sum()))
print("Sparse ratio:", f"{generation_summary['sparse_points'].sum() / generation_summary['full_points'].sum():.2%}")

agent_timing_cols = ["init_time", "dest_time", "traj_time", "sample_time", "extract_time", "agent_time"]
agent_timing = generation_summary[agent_timing_cols].sum().sort_values(ascending=False)
print("\nAgent CPU seconds (summed across agents)")
print(agent_timing.to_string())
print("\nAgent CPU share")
print((agent_timing.drop("agent_time") / agent_timing["agent_time"]).sort_values(ascending=False).map("{:.1%}".format).to_string())

batch_timing_cols = ["city_load_time", "agent_loop_time", "concat_time", "reproject_time", "write_time", "batch_time"]
print("\nBatch wall (max across workers)")
print(batch_timing[batch_timing_cols].max().sort_values(ascending=False).to_string())
print("\nBatch CPU seconds (summed across workers)")
print(batch_timing[batch_timing_cols].sum().sort_values(ascending=False).to_string())

generation_summary[agent_timing_cols + ["full_points", "sparse_points", "dest_entries", "diary_entries"]].describe()

Sparse rows: 239023
Diary rows: 91971
Destination diary rows: 80205
Sparse ratio: 1.06%


,dest_entries,diary_entries,full_points,sparse_points,beta_start,beta_ping,beta_durations,ha
count,200.000000,200.000000,200.0,200.000000,200.0,200.0,200.0,2.000000e+02
mean,401.025000,459.855000,112321.0,1195.115000,300.0,7.0,55.0,7.666667e-01
std,21.273627,41.123835,0.0,122.015164,0.0,0.0,0.0,1.113009e-16
min,336.000000,343.000000,112321.0,833.000000,300.0,7.0,55.0,7.666667e-01
25%,386.000000,433.000000,112321.0,1103.750000,300.0,7.0,55.0,7.666667e-01
50%,401.000000,459.500000,112321.0,1179.000000,300.0,7.0,55.0,7.666667e-01
75%,415.250000,479.500000,112321.0,1282.750000,300.0,7.0,55.0,7.666667e-01
max,452.000000,623.000000,112321.0,1523.000000,300.0,7.0,55.0,7.666667e-01
